# Inference and resolution

_Artificial Intelligence (CS221)_

**Mechanical rules that crank out new true facts from old ones.**

Checking truth tables is slow when there are many symbols.

     Inference rules derive new facts directly, by pattern. No table needed.

---

This notebook is a practice scaffold for the **Inference and resolution** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python

In [ ]:
# Rules as (premises, conclusion). Definite clauses (Horn).
rules = [
    (["A", "B"], "C"),
    (["C"], "D"),
    (["A"], "B"),
    (["D", "C"], "E"),
]
facts = {"A"}                            # known facts to start

changed = True
while changed:                           # keep firing rules until nothing new
    changed = False
    for premises, conclusion in rules:
        if conclusion not in facts and all(p in facts for p in premises):
            facts.add(conclusion)        # modus ponens fires
            print("derived", conclusion, "from", premises)
            changed = True

print("final known facts:", sorted(facts))
print("E proven?", "E" in facts)

## Visualize the data & results

_Starting from family facts about Tom, Bob, Ann and Pat, which new relationships does forward chaining derive?_

In [ ]:
import matplotlib.pyplot as plt

facts = {("parent","Tom","Bob"), ("parent","Bob","Ann"),
         ("parent","Bob","Pat"), ("male","Tom"), ("male","Bob")}
order = []
changed = True
while changed:
    changed = False
    for f1 in list(facts):                 # grandparent(X,Z) :- parent(X,Y), parent(Y,Z)
        if f1[0] == "parent":
            for f2 in list(facts):
                if f2[0] == "parent" and f1[2] == f2[1]:
                    gp = ("grandparent", f1[1], f2[2])
                    if gp not in facts: facts.add(gp); order.append(gp); changed = True
    for f in list(facts):                  # grandfather(X,Z) :- grandparent(X,Z), male(X)
        if f[0] == "grandparent" and ("male", f[1]) in facts:
            gf = ("grandfather", f[1], f[2])
            if gf not in facts: facts.add(gf); order.append(gf); changed = True
print("derived", order)

labels = ["parent facts", "gp(Tom,Ann)", "gp(Tom,Pat)", "gf(Tom,Ann)", "gf(Tom,Pat)"]
cols = ["#4ea1ff", "#7ee787", "#7ee787", "#ffb454", "#ffb454"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, [1]*5, color=cols)
tags = ["given","derived","derived","derived","derived"]
for i, t in enumerate(tags): ax.text(i, 1, t, ha="center", va="bottom")
ax.set_ylim(0, 1.4)
ax.set_title("family facts before vs after forward chaining")
plt.show()